# indah in a notebook

indah's home turf is a hosted notebook (Colab, Runpod). This notebook runs the
same thing **locally**, so you can try the inline experience without a cloud
runtime.

Run the cell below: it builds a small reactive UI and embeds it right in the
notebook. Type a prompt and click **Generate** - a mock LLM streams its reply
into the page token by token over SSE, while the slider stays fully responsive
(async work never freezes the UI). Drag the slider mid-stream to see for yourself.

In [ ]:
import asyncio

import indah
from indah import (
    Button,
    Column,
    Session,
    Signal,
    Slider,
    StreamText,
    Text,
    TextInput,
    computed,
    create_app,
)


# The "LLM" is a plain async generator with no indah imports, so a real model
# client drops straight in behind the same shape (ADR-0009).
async def mock_llm(prompt):
    reply = f"You asked '{prompt.strip()}'. Here is a streamed reply, token by token."
    for word in reply.split(" "):
        await asyncio.sleep(0.08)
        yield word + " "


prompt = Signal("")
stream = StreamText(label="Response")


# An async handler: it awaits the generator and feeds each token into the UI.
async def on_generate():
    stream.reset()
    async for token in mock_llm(prompt.value):
        stream.feed(token)


# A slider + label, unrelated to the stream, to prove the UI stays live.
a = Signal(3)
doubled = computed(lambda: f"the slider stays live while streaming: 2 x {a.value} = {2 * a.value}")

page = Column(
    children=[
        Text("indah: async token streaming"),
        TextInput(prompt, placeholder="Ask the mock LLM something...", label="Prompt"),
        Button("Generate", on_click=on_generate),
        stream,
        Slider(a, min=0, max=10, label="a"),
        Text(doubled),
    ]
)

# In a notebook, launch() serves on a background thread and embeds the app inline.
handle = indah.launch(create_app(session=Session(page)), block=False)

The iframe above is the live app. Type a prompt and click **Generate**: tokens
arrive incrementally as *append* patches (not one final dump), and only the output
node grows. Drag the slider while it streams - the label re-computes immediately,
because the generation runs as a background task and never blocks the event loop.

When you're done, stop the server:

In [ ]:
handle.stop()